# Volume 1. Noise And Pattern Completion

Question: when an assembly is partially cued, does recurrence recover the original sparse set or drift somewhere else?

This notebook deliberately damages a trained assembly at several levels. Recovery is measured by overlap with the reference assembly.


In [ ]:
import copy

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import HTML

from neural_assemblies.assembly_calculus.tracing import pattern_complete_trace, project_trace
from neural_assemblies.core.brain import Brain
from neural_assemblies.viz import animate_assembly_trace, plot_assemblies, plot_parameter_heatmap, plot_winner_turnover


In [ ]:
N = 5_000
K = 80
BETA = 0.10
ROUNDS = 9
FRACTIONS = [0.20, 0.35, 0.50, 0.75]

brain = Brain(p=0.05, save_winners=True, seed=31, engine="numpy_sparse")
brain.add_stimulus("red", K)
brain.add_area("COLOR", N, K, BETA)
training_trace = project_trace(brain, "red", "COLOR", rounds=ROUNDS)
reference = training_trace.final

pd.DataFrame([training_trace.summary()])


Run independent completion trials from different retained fractions. Each trial starts from the same trained brain snapshot.


In [ ]:
base_brain = copy.deepcopy(brain)
diagnostics = []
for fraction in FRACTIONS:
    trial_brain = copy.deepcopy(base_brain)
    diagnostics.append(
        pattern_complete_trace(trial_brain, "COLOR", fraction=fraction, rounds=6, seed=5)
    )

pd.DataFrame([diagnostic.to_record() for diagnostic in diagnostics])


In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
records = pd.DataFrame([diagnostic.to_record() for diagnostic in diagnostics])
bars = ax.bar(records["kept_fraction"].astype(str), records["recovery_overlap"], color="#7768ae")
ax.axhline(records["chance_baseline"].iloc[0], color="#555555", linestyle="--", label="chance baseline")
ax.bar_label(bars, labels=[f"{value:.2f}" for value in records["recovery_overlap"]], padding=3)
ax.set_ylim(0, 1.05)
ax.set_xlabel("fraction of winners retained")
ax.set_ylabel("overlap with reference")
ax.set_title("Pattern completion recovery")
ax.legend(frameon=False)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.show()
plt.close(fig)


Inspect one middle case closely: reference, partial cue, and final recovered assembly.


In [ ]:
mid = diagnostics[2]
fig, axes = plot_assemblies(
    [mid.reference, mid.partial, mid.final],
    N,
    titles=["reference", "partial cue", "after recurrence"],
    subtitles=[
        f"{len(mid.reference)} winners",
        f"{len(mid.partial)} retained",
        f"overlap {mid.recovery_overlap:.2f}",
    ],
    colors=["#4d9de0", "#f2c14e", "#59a14f"],
    marker_size=16,
)
plt.show()
plt.close(fig)


In [ ]:
fig, animation = animate_assembly_trace(
    mid.trace,
    N,
    title="Pattern completion from a partial cue",
    color="#7768ae",
    interval=750,
)
html = HTML(animation.to_jshtml())
plt.close(fig)
html


In [ ]:
ax, matrix = plot_parameter_heatmap(
    records[["recovery_overlap"]].T.values,
    x_labels=records["kept_fraction"],
    y_labels=["recovery"],
    title="Recovery by retained fraction",
    cbar_label="overlap",
)
ax.set_xlabel("fraction retained")
plt.show()
plt.close(ax.figure)


In [ ]:
ax, turnover = plot_winner_turnover(mid.trace, title="Winner turnover during completion")
plt.show()
plt.close(ax.figure)


The boundary case is the point: recurrence can repair partial cues only when the trained attractor and retained signal are strong enough. The diagnostics show when recovery is high, marginal, or mostly drift.
